In [1]:
# CÉLULA 1 — Instalação de dependências
!pip install transformers torch pandas openpyxl scikit-learn -q

print('✅ Dependências instaladas!')

✅ Dependências instaladas!


In [2]:
# CÉLULA 2 — Imports e montagem do Drive
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import drive

drive.mount('/content/drive')

print('✅ Pronto!')

Mounted at /content/drive
✅ Pronto!


In [4]:
# CÉLULA 3 — Carregar e fazer merge dos ground truths v1 e v2

CAMINHO_V1 = '/content/ground_truth.csv'
CAMINHO_V2 = '/content/ground_truth_v2.xlsx'

# Carregar v1
df_v1 = pd.read_csv(CAMINHO_V1)
print(f'Ground truth v1: {len(df_v1)} avaliações')

# Carregar v2
df_v2 = pd.read_excel(CAMINHO_V2)
print(f'Ground truth v2: {len(df_v2)} avaliações')

# Garantir mesmas colunas
colunas = ['texto', 'sentimento', 'aspecto', 'sentimento_aspecto', 'plataforma', 'data_coleta', 'fonte_promocao', 'produto']
df_v1 = df_v1[colunas]
df_v2 = df_v2[colunas]

# Merge
df = pd.concat([df_v1, df_v2], ignore_index=True)
print(f'\nTotal após merge: {len(df)} avaliações')

# Limpeza: padronizar sentimento_aspecto
def limpar_label(label):
    if pd.isna(label):
        return label
    label = str(label).strip().lower()
    for sufixo in ['_qualidade', '_entrega', '_preço', '_preco']:
        label = label.replace(sufixo, '')
    return label

df['sentimento_aspecto'] = df['sentimento_aspecto'].apply(limpar_label)
df['sentimento'] = df['sentimento'].apply(limpar_label)

# Filtrar apenas os 3 aspectos suportados
ASPECTOS_VALIDOS = ['preço', 'entrega', 'qualidade']
df = df[df['aspecto'].isin(ASPECTOS_VALIDOS)].reset_index(drop=True)

# Remover labels inválidos
LABELS_VALIDOS = ['positivo', 'negativo', 'neutro']
df = df[df['sentimento_aspecto'].isin(LABELS_VALIDOS)].reset_index(drop=True)

print(f'\nApós filtro de aspectos e limpeza: {len(df)} avaliações')
print('\nAspectos:')
print(df['aspecto'].value_counts())
print('\nSentimento aspecto (gabarito):')
print(df['sentimento_aspecto'].value_counts())

Ground truth v1: 110 avaliações
Ground truth v2: 360 avaliações

Total após merge: 470 avaliações

Após filtro de aspectos e limpeza: 418 avaliações

Aspectos:
aspecto
qualidade    298
entrega       76
preço         44
Name: count, dtype: int64

Sentimento aspecto (gabarito):
sentimento_aspecto
positivo    221
negativo    119
neutro       78
Name: count, dtype: int64


In [5]:
# CÉLULA 4 — Carregar o modelo v2 do HuggingFace

MODELO_HF = 'Amand4priscil4/promosense-modelo'

print(f'Carregando modelo: {MODELO_HF}')
tokenizer = AutoTokenizer.from_pretrained(MODELO_HF)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO_HF)

# Verificar labels do modelo
print(f'\nLabels do modelo: {modelo.config.id2label}')

# Usar GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo.to(device)
print(f'\nDispositivo: {device}')
print('✅ Modelo carregado!')

Carregando modelo: Amand4priscil4/promosense-modelo


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Labels do modelo: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}

Dispositivo: cpu
✅ Modelo carregado!


In [7]:
# CÉLULA 4.1 — Descobrir mapeamento dos labels
textos_teste = [
    'Produto ótimo, adorei, excelente qualidade!',  # esperado: positivo
    'Péssimo produto, veio quebrado, horrível!',     # esperado: negativo
    'Produto ok, nada demais, chegou no prazo.'      # esperado: neutro
]

for texto in textos_teste:
    inputs = tokenizer(texto, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = modelo(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    print(f'LABEL_{pred} → "{texto[:50]}"')

LABEL_2 → "Produto ótimo, adorei, excelente qualidade!"
LABEL_0 → "Péssimo produto, veio quebrado, horrível!"
LABEL_2 → "Produto ok, nada demais, chegou no prazo."


In [6]:
# CÉLULA 4.2 — Achar LABEL_1
textos_teste2 = [
    'Chegou no prazo, produto ok, sem reclamações.',
    'Mais ou menos, esperava melhor pelo preço.',
    'Produto razoável, nada demais.',
    'Demorou um pouco mas chegou tudo certo.'
]

for texto in textos_teste2:
    inputs = tokenizer(texto, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = modelo(**inputs)
    logits = outputs.logits
    pred = torch.argmax(logits, dim=-1).item()
    probs = torch.softmax(logits, dim=-1)[0]
    print(f'LABEL_{pred} (0:{probs[0]:.2f} 1:{probs[1]:.2f} 2:{probs[2]:.2f}) → "{texto[:50]}"')

LABEL_2 (0:0.00 1:0.02 2:0.98) → "Chegou no prazo, produto ok, sem reclamações."
LABEL_1 (0:0.31 1:0.38 2:0.31) → "Mais ou menos, esperava melhor pelo preço."
LABEL_2 (0:0.03 1:0.08 2:0.89) → "Produto razoável, nada demais."
LABEL_2 (0:0.02 1:0.09 2:0.88) → "Demorou um pouco mas chegou tudo certo."


In [8]:
# CÉLULA 5 — Léxico completo do PromoSense (sentimento + aspecto)

# --- Léxico de sentimento (do projeto original) ---
lexico_sentimento = {
    # NEGATIVAS
    'atrasou': -1, 'não chegou': -1, 'extraviado': -1,
    'demorou demais': -1, 'atrasado': -1, 'atrasada': -1,
    'demorou': -1, 'extraviada': -1, 'perdido': -1,
    'veio errado': -1, 'veio quebrado': -1, 'veio faltando': -1,
    'veio amassado': -1, 'veio sujo': -1,
    'não funcionou': -1, 'parou de funcionar': -1,
    'produto falhou': -1, 'estourou': -1, 'defeito': -1,
    'falsificado': -1, 'não é original': -1,
    'embalagem danificada': -1, 'produto danificado': -1,
    'cor diferente': -1, 'tamanho errado': -1,
    'muito pequeno': -1, 'muito grande': -1,
    'muito fino': -1, 'não aguentou': -1, 'frágil': -1,
    'péssima qualidade': -1, 'deixando a desejar': -1,
    'cheiro forte': -1, 'cheiro ruim': -1,
    'material fraco': -1, 'material ruim': -1,
    'acabamento ruim': -1, 'acabamento péssimo': -1,
    'tinta saindo': -1, 'colou mal': -1,
    'foto enganosa': -1, 'propaganda enganosa': -1,
    'não vale o preço': -1, 'caro demais': -1,
    'péssimo': -1, 'péssima': -1, 'horrível': -1, 'terrível': -1,
    'ruim': -1, 'fraco': -1, 'fraca': -1,
    'quebrado': -1, 'quebrada': -1,
    'defeituoso': -1, 'defeituosa': -1,
    'falso': -1, 'falsa': -1,
    'não gostei': -1, 'não recomendo': -1,
    'decepcionado': -1, 'decepcionada': -1, 'decepcionante': -1,
    'frustrado': -1, 'frustrada': -1,
    'arrependido': -1, 'arrependida': -1,
    'insatisfeito': -1, 'insatisfeita': -1,
    'desapontado': -1, 'desapontada': -1,
    'lamentável': -1, 'inadmissível': -1,
    'absurdo': -1, 'absurda': -1,
    'devolvei': -1, 'pedi reembolso': -1, 'abri reclamação': -1,
    'amassado': -1, 'amassada': -1,
    'riscado': -1, 'riscada': -1,
    'sujo': -1, 'suja': -1,
    'estragado': -1, 'estragada': -1,
    'vazando': -1, 'vazou': -1,
    'manchado': -1, 'manchada': -1,
    'errado': -1, 'errada': -1,
    'faltando': -1, 'incompleto': -1, 'incompleta': -1,
    'descaso': -1, 'negligente': -1,
    'grosseiro': -1, 'grosseira': -1,
    'só não gostei': -1, 'imaginei ser mais': -1, 'mais resistente': -1,

    # POSITIVAS
    'chegou rápido': 1, 'chegou antes': 1,
    'entrega rápida': 1, 'entrega perfeita': 1,
    'antes do prazo': 1,
    'bem embalado': 1, 'muito bem embalado': 1,
    'embalagem perfeita': 1, 'embalagem ótima': 1,
    'como descrito': 1, 'igual ao anúncio': 1,
    'exatamente como descrito': 1, 'conforme anunciado': 1,
    'original': 1, 'produto novo': 1, 'produto top': 1,
    'funcionou perfeitamente': 1, 'veio certinho': 1,
    'boa qualidade': 1, 'muito bom': 1,
    'superou expectativas': 1, 'melhor que esperado': 1,
    'superou o esperado': 1, 'custo benefício': 1, 'valeu a pena': 1,
    'ótimo': 1, 'ótima': 1, 'excelente': 1,
    'perfeito': 1, 'perfeita': 1,
    'maravilhoso': 1, 'maravilhosa': 1,
    'incrível': 1, 'fantástico': 1, 'fantástica': 1,
    'bom': 1, 'boa': 1, 'bonito': 1, 'bonita': 1,
    'lindo': 1, 'linda': 1,
    'resistente': 1, 'durável': 1,
    'macio': 1, 'macia': 1, 'confortável': 1,
    'prático': 1, 'prática': 1, 'versátil': 1,
    'produto incrível': 1,
    'gostei': 1, 'amei': 1, 'adorei': 1,
    'amei o produto': 1, 'recomendo': 1,
    'super recomendo': 1, 'comprarei novamente': 1,
    'voltarei a comprar': 1,
    'satisfeito': 1, 'satisfeita': 1, 'feliz': 1,
    'agradou': 1, 'superou': 1, 'valeu': 1, 'compensou': 1,
    'rápido': 1, 'rápida': 1, 'pontual': 1,
    'atencioso': 1, 'atenciosa': 1, 'eficiente': 1,
    'correto': 1, 'correta': 1,
    'caprichado': 1, 'caprichada': 1,
    'atendimento excelente': 1, 'vendedor confiável': 1,

    # NEUTRAS
    'produto ok': 0, 'nada demais': 0,
    'dentro do esperado': 0, 'mais ou menos': 0,
    'razoável': 0, 'aparentemente': 0,
    'deve durar com cuidados': 0, 'serve para o uso': 0,
    'cumpre o propósito': 0, 'nem bom nem ruim': 0,
    'poderia ser melhor': 0, 'aceitável': 0,
    'ok': 0, 'normal': 0, 'mediano': 0, 'mediana': 0,
    'regular': 0, 'simples': 0,
    'básico': 0, 'básica': 0, 'comum': 0, 'padrão': 0,
}

# --- Palavras-chave por aspecto ---
LEXICO_ASPECTOS = {
    'preço': [
        'preço', 'preco', 'valor', 'custo', 'caro', 'barato',
        'custo benefício', 'custo-benefício', 'custo beneficio',
        'vale a pena', 'investimento', 'paguei', 'cobram',
        'acessível', 'acessivel', 'econômico', 'economico'
    ],
    'entrega': [
        'entrega', 'chegou', 'prazo', 'frete', 'envio', 'enviou',
        'demorou', 'rápido', 'rapido', 'atrasou', 'atraso',
        'antes do prazo', 'no prazo', 'postagem', 'transportadora',
        'correios', 'entregador', 'embalado', 'embalagem'
    ],
    'qualidade': [
        'qualidade', 'produto', 'material', 'funciona', 'funcionando',
        'original', 'defeito', 'quebrou', 'danificado', 'resistente',
        'durável', 'duravel', 'acabamento', 'potente', 'bateria',
        'som', 'tela', 'desempenho'
    ]
}

# --- Mapeamento de labels ---
MAPA_LABEL = {1: 'positivo', -1: 'negativo', 0: 'neutro'}
MAPA_MODELO = {
    'positive': 'positivo', 'negative': 'negativo', 'neutral': 'neutro',
    'positivo': 'positivo', 'negativo': 'negativo', 'neutro': 'neutro',
    'label_0': 'negativo', 'label_1': 'neutro', 'label_2': 'positivo'
}

def detectar_aspecto(texto):
    """Detecta o aspecto dominante por palavras-chave."""
    texto_lower = texto.lower()
    contagem = {asp: 0 for asp in LEXICO_ASPECTOS}
    for asp, palavras in LEXICO_ASPECTOS.items():
        for palavra in palavras:
            if palavra in texto_lower:
                contagem[asp] += 1
    aspecto = max(contagem, key=contagem.get)
    return aspecto if contagem[aspecto] > 0 else 'qualidade'

def classificar_sentimento_hibrido(texto):
    """
    Classifica sentimento usando léxico primeiro,
    BERTimbau como fallback quando léxico não cobre.
    """
    texto_lower = texto.lower()

    # Ordenar expressões por tamanho (priorizar frases sobre palavras)
    expressoes = sorted(lexico_sentimento.keys(), key=len, reverse=True)

    scores = []
    for expr in expressoes:
        if expr in texto_lower:
            scores.append(lexico_sentimento[expr])

    if scores:
        # Léxico encontrou expressões — usa média dos scores
        media = sum(scores) / len(scores)
        if media > 0.2:
            return 'positivo', 'lexico'
        elif media < -0.2:
            return 'negativo', 'lexico'
        else:
            return 'neutro', 'lexico'
    else:
        # Fallback: BERTimbau
        inputs = tokenizer(
            texto,
            return_tensors='pt',
            truncation=True,
            max_length=128,
            padding=True
        ).to(device)
        with torch.no_grad():
            outputs = modelo(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        label = modelo.config.id2label[pred].lower()
        return MAPA_MODELO.get(label, label), 'bertimbau'

# --- Teste ---
exemplos_teste = [
    ('O produto chegou antes do prazo, entrega excelente!', 'positivo'),
    ('Produto ok, nada demais, chegou no prazo.', 'neutro'),
    ('Veio quebrado, péssima qualidade, não recomendo.', 'negativo'),
    ('Mais ou menos, esperava melhor pelo preço.', 'neutro'),
]

print('Teste da classificação híbrida:')
print(f'{"Texto":<55} {"Esperado":<12} {"Previsto":<12} {"Fonte"}')
print('-' * 95)
for texto, esperado in exemplos_teste:
    aspecto = detectar_aspecto(texto)
    sentimento, fonte = classificar_sentimento_hibrido(texto)
    ok = '✅' if sentimento == esperado else '❌'
    print(f'{ok} {texto[:50]:<53} {esperado:<12} {sentimento:<12} [{fonte}]')

print('\n✅ Léxico híbrido pronto!')

Teste da classificação híbrida:
Texto                                                   Esperado     Previsto     Fonte
-----------------------------------------------------------------------------------------------
✅ O produto chegou antes do prazo, entrega excelente    positivo     positivo     [lexico]
✅ Produto ok, nada demais, chegou no prazo.             neutro       neutro       [lexico]
✅ Veio quebrado, péssima qualidade, não recomendo.      negativo     negativo     [lexico]
✅ Mais ou menos, esperava melhor pelo preço.            neutro       neutro       [lexico]

✅ Léxico híbrido pronto!


In [9]:
# CÉLULA 6 — Inferência em lote com abordagem híbrida

def inferencia_em_lote_hibrida(textos, batch_size=32):
    """Roda classificação híbrida em lote."""
    sentimentos = []
    fontes = []

    # Separar textos que precisam do BERTimbau
    indices_bertimbau = []
    for i, texto in enumerate(textos):
        sent, fonte = classificar_sentimento_hibrido(texto)
        sentimentos.append(sent)
        fontes.append(fonte)
        if fonte == 'bertimbau':
            indices_bertimbau.append(i)

    print(f'  Léxico resolveu: {fontes.count("lexico")}/{len(textos)}')
    print(f'  BERTimbau usado: {fontes.count("bertimbau")}/{len(textos)}')

    return sentimentos, fontes

print(f'Rodando inferência híbrida em {len(df)} avaliações...')
sentimentos, fontes = inferencia_em_lote_hibrida(df['texto'].tolist())

df['sentimento_previsto'] = sentimentos
df['fonte_classificacao'] = fontes

print('\n✅ Inferência concluída!')
print('\nDistribuição dos sentimentos previstos:')
print(df['sentimento_previsto'].value_counts())

Rodando inferência híbrida em 418 avaliações...
  Léxico resolveu: 352/418
  BERTimbau usado: 66/418

✅ Inferência concluída!

Distribuição dos sentimentos previstos:
sentimento_previsto
positivo    301
negativo     76
neutro       41
Name: count, dtype: int64


In [10]:
# CÉLULA 7 — Detecção de aspecto e métricas completas

print('Detectando aspectos...')
df['aspecto_detectado'] = df['texto'].apply(lambda x: detectar_aspecto(str(x)))

print('✅ Aspectos detectados!')
print('\nDistribuição dos aspectos detectados:')
print(df['aspecto_detectado'].value_counts())

print('\n' + '=' * 60)
print('MÉTRICAS — PromoSense ABSA Híbrida')
print('=' * 60)

# --- Sentimento geral ---
print('\n📊 SENTIMENTO GERAL:')
print(classification_report(
    df['sentimento_aspecto'],
    df['sentimento_previsto'],
    labels=['positivo', 'negativo', 'neutro'],
    target_names=['Positivo', 'Negativo', 'Neutro']
))

acertos = (df['sentimento_aspecto'] == df['sentimento_previsto']).sum()
print(f'Accuracy geral: {acertos/len(df):.1%} ({acertos}/{len(df)})')

# --- Detecção de aspecto ---
print('\n📊 DETECÇÃO DE ASPECTO:')
acertos_asp = (df['aspecto'] == df['aspecto_detectado']).sum()
print(f'Accuracy aspecto: {acertos_asp/len(df):.1%} ({acertos_asp}/{len(df)})')
print()
print(classification_report(
    df['aspecto'],
    df['aspecto_detectado'],
    labels=['preço', 'entrega', 'qualidade'],
    target_names=['Preço', 'Entrega', 'Qualidade']
))

# --- Sentimento por aspecto ---
print('\n📊 SENTIMENTO POR ASPECTO:')
for aspecto in ['preço', 'entrega', 'qualidade']:
    df_asp = df[df['aspecto'] == aspecto]
    acc = (df_asp['sentimento_aspecto'] == df_asp['sentimento_previsto']).mean()
    print(f'  {aspecto.capitalize():10s}: {acc:.1%} ({len(df_asp)} amostras)')

Detectando aspectos...
✅ Aspectos detectados!

Distribuição dos aspectos detectados:
aspecto_detectado
qualidade    236
entrega      149
preço         33
Name: count, dtype: int64

MÉTRICAS — PromoSense ABSA Híbrida

📊 SENTIMENTO GERAL:
              precision    recall  f1-score   support

    Positivo       0.71      0.96      0.82       221
    Negativo       0.76      0.49      0.59       119
      Neutro       0.34      0.18      0.24        78

    accuracy                           0.68       418
   macro avg       0.60      0.54      0.55       418
weighted avg       0.66      0.68      0.64       418

Accuracy geral: 68.2% (285/418)

📊 DETECÇÃO DE ASPECTO:
Accuracy aspecto: 74.2% (310/418)

              precision    recall  f1-score   support

       Preço       0.82      0.61      0.70        44
     Entrega       0.44      0.87      0.59        76
   Qualidade       0.92      0.73      0.81       298

    accuracy                           0.74       418
   macro avg       

In [12]:
import os
os.makedirs('/content/drive/MyDrive/PromoSense', exist_ok=True)

In [13]:
# CÉLULA 8 — Salvar resultados

CAMINHO_SAIDA = '/content/drive/MyDrive/PromoSense/resultado_absa_hibrida.csv'

df['absa_resultado'] = df['aspecto_detectado'] + ':' + df['sentimento_previsto']
df['absa_correto'] = (
    (df['aspecto'] == df['aspecto_detectado']) &
    (df['sentimento_aspecto'] == df['sentimento_previsto'])
)

colunas_saida = [
    'texto', 'produto', 'fonte_promocao',
    'aspecto', 'aspecto_detectado',
    'sentimento_aspecto', 'sentimento_previsto',
    'fonte_classificacao', 'absa_resultado', 'absa_correto'
]

df[colunas_saida].to_csv(CAMINHO_SAIDA, index=False, encoding='utf-8-sig')

print(f'✅ Resultados salvos em: {CAMINHO_SAIDA}')
print(f'\nResumo por aspecto:')
resumo = df.groupby('aspecto')['absa_correto'].agg(['sum', 'count'])
resumo.columns = ['acertos', 'total']
resumo['accuracy'] = (resumo['acertos'] / resumo['total']).map('{:.1%}'.format)
print(resumo)

print(f'\nAccuracy ABSA completa (aspecto + sentimento corretos):')
print(f"{df['absa_correto'].mean():.1%} ({df['absa_correto'].sum()}/{len(df)})")

✅ Resultados salvos em: /content/drive/MyDrive/PromoSense/resultado_absa_hibrida.csv

Resumo por aspecto:
           acertos  total accuracy
aspecto                           
entrega         46     76    60.5%
preço           23     44    52.3%
qualidade      151    298    50.7%

Accuracy ABSA completa (aspecto + sentimento corretos):
52.6% (220/418)


In [ ]:
# CÉLULA 9 — Salvar no repositório

!git config --global user.email "amanda.priscilaa15@gmail.com"
!git config --global user.name "amand4priscil4"

!git clone https://SEU_TOKEN@github.com/amand4priscil4/PROMOSENSE.git /content/PROMOSENSE

!cp /content/drive/MyDrive/PromoSense/resultado_absa_hibrida.csv /content/PROMOSENSE/data/processed/

%cd /content/PROMOSENSE
!git add data/processed/resultado_absa_hibrida.csv
!git commit -m "feat: ABSA híbrida (léxico + BERTimbau) - accuracy 52.6%"
!git push

Cloning into '/content/PROMOSENSE'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 44 (delta 13), reused 37 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 882.69 KiB | 7.00 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/PROMOSENSE
[main 5dcaf3e] feat: ABSA híbrida (léxico + BERTimbau) - accuracy 52.6%
 1 file changed, 419 insertions(+)
 create mode 100644 data/processed/resultado_absa_hibrida.csv
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 24.62 KiB | 2.73 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/amand4priscil4/PROMOSENSE.git
   5f21456..5dcaf3e  main -> main
